In [6]:
from networkx.classes import edges
from sqlalchemy import create_engine
import geopandas as gpd
import osmnx as ox
import networkx as nx
from dotenv import load_dotenv
import os

load_dotenv()

True

In [27]:
G = ox.graph_from_place('Zurich, Switzerland', network_type='walk')
#nodes, edges = ox.graph_to_gdfs(G)

In [29]:
G = ox.add_node_elevations_google(G, api_key=os.getenv('GOOGLE_API_KEY'))

In [55]:
nodes, edges = ox.graph_to_gdfs(G)

In [51]:
edges.head()

osmid  oneway lanes              name      highway  \
u      v          key                                                           
453805 3780703398 0    172063131   False     3       Hohlstrasse     tertiary   
       3780703400 0    374688190   False     3       Hohlstrasse     tertiary   
       3780703401 0      5880367   False     4  Duttweilerbrücke     tertiary   
       5730016968 0    521343968   False     2    Herdernstrasse     tertiary   
453810 3835970612 0    521343964   False   NaN     Baslerstrasse  residential   

                      maxspeed reversed  length  \
u      v          key                             
453805 3780703398 0         50    False  13.977   
       3780703400 0         50     True  19.077   
       3780703401 0         50     True  17.984   
       5730016968 0         50    False  14.136   
453810 3835970612 0         30     True  13.086   

                                                                geometry  \
u      v          key                                                      
453805 3780703398 0      LINESTRING (8.50840 47.38508, 8.50857 47.38502)   
       3780703400 0      LINESTRING (8.50840 47.38508, 8.50817 47.38515)   
       3780703401 0    LINESTRING (8.50840 47.38508, 8.50848 47.38516...   
       5730016968 0    LINESTRING (8.50840 47.38508, 8.50833 47.38501...   
453810 3835970612 0      LINESTRING (8.50608 47.38322, 8.50594 47.38329)   

                      service  ref bridge access width junction tunnel  \
u      v          key                                                    
453805 3780703398 0       NaN  NaN    NaN    NaN   NaN      NaN    NaN   
       3780703400 0       NaN  NaN    NaN    NaN   NaN      NaN    NaN   
       3780703401 0       NaN  NaN    NaN    NaN   NaN      NaN    NaN   
       5730016968 0       NaN  NaN    NaN    NaN   NaN      NaN    NaN   
453810 3835970612 0       NaN  NaN    NaN    NaN   NaN      NaN    NaN   

                      est_width area  delta_elevation  
u      v          key                                  
453805 3780703398 0         NaN  NaN           -0.140  
       3780703400 0         NaN  NaN           -0.127  
       3780703401 0         NaN  NaN           -0.122  
       5730016968 0         NaN  NaN           -0.018  
453810 3835970612 0         NaN  NaN           -0.119

In [56]:
for row, values in edges.iterrows():
    u, v, _ = row
    edges.at[row, 'gain'] = abs((nodes.loc[v, 'elevation'] - nodes.loc[u, 'elevation']) / values['length'])

In [54]:
edges.head()

osmid  oneway lanes              name      highway  \
u      v          key                                                           
453805 3780703398 0    172063131   False     3       Hohlstrasse     tertiary   
       3780703400 0    374688190   False     3       Hohlstrasse     tertiary   
       3780703401 0      5880367   False     4  Duttweilerbrücke     tertiary   
       5730016968 0    521343968   False     2    Herdernstrasse     tertiary   
453810 3835970612 0    521343964   False   NaN     Baslerstrasse  residential   

                      maxspeed reversed  length  \
u      v          key                             
453805 3780703398 0         50    False  13.977   
       3780703400 0         50     True  19.077   
       3780703401 0         50     True  17.984   
       5730016968 0         50    False  14.136   
453810 3835970612 0         30     True  13.086   

                                                                geometry  \
u      v          key                                                      
453805 3780703398 0      LINESTRING (8.50840 47.38508, 8.50857 47.38502)   
       3780703400 0      LINESTRING (8.50840 47.38508, 8.50817 47.38515)   
       3780703401 0    LINESTRING (8.50840 47.38508, 8.50848 47.38516...   
       5730016968 0    LINESTRING (8.50840 47.38508, 8.50833 47.38501...   
453810 3835970612 0      LINESTRING (8.50608 47.38322, 8.50594 47.38329)   

                      service  ref bridge access width junction tunnel  \
u      v          key                                                    
453805 3780703398 0       NaN  NaN    NaN    NaN   NaN      NaN    NaN   
       3780703400 0       NaN  NaN    NaN    NaN   NaN      NaN    NaN   
       3780703401 0       NaN  NaN    NaN    NaN   NaN      NaN    NaN   
       5730016968 0       NaN  NaN    NaN    NaN   NaN      NaN    NaN   
453810 3835970612 0       NaN  NaN    NaN    NaN   NaN      NaN    NaN   

                      est_width area  elevation  
u      v          key                            
453805 3780703398 0         NaN  NaN   0.010016  
       3780703400 0         NaN  NaN   0.006657  
       3780703401 0         NaN  NaN   0.006784  
       5730016968 0         NaN  NaN   0.001273  
453810 3835970612 0         NaN  NaN   0.009094

In [7]:
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}")

In [59]:
nodes.to_postgis('zurich_nodes', engine, if_exists='replace', index=True, schema='gta_p1')
edges.to_postgis('zurich_edges', engine, if_exists='replace', index=True, schema='gta_p1')

In [60]:
import secrets
secret_key = secrets.token_hex(16)

In [61]:
print(secret_key)

c025053bc32b33d1cf038184905c1220
